# Ablation Study (Phase 6)

Compare final system against no-MST and no-fairness ablations.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
ROOT = Path.cwd().parent
metrics_dir = ROOT / 'outputs' / 'metrics'
split = 'dev'

experiments = [
    'kg2rag_fair',
    'ablation_no_mst',
    'ablation_no_fairness',
]

rows = []
for exp in experiments:
    path = metrics_dir / f'{exp}_metrics_{split}.json'
    if not path.exists():
        print(f'Missing metrics for {exp}: {path}')
        continue

    payload = pd.read_json(path, typ='series')
    fairness = payload.get('fairness', {}).get('geo_group', {})

    rows.append({
        'experiment': exp,
        'exact_match': payload.get('accuracy', {}).get('exact_match'),
        'answer_f1': payload.get('accuracy', {}).get('answer_f1'),
        'mrr': payload.get('retrieval', {}).get('mrr'),
        'parity_diff_geo': fairness.get('demographic_parity_diff'),
        'eq_odds_geo': fairness.get('equalized_odds_diff'),
    })

ablation_df = pd.DataFrame(rows)
ablation_df

In [ ]:
if not ablation_df.empty:
    metric_cols = ['exact_match', 'answer_f1', 'mrr']
    plot_df = ablation_df.melt(id_vars=['experiment'], value_vars=metric_cols, var_name='metric', value_name='value')

    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x='metric', y='value', hue='experiment')
    plt.title('Ablation Performance Comparison')
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print('No ablation metrics found yet.')

In [ ]:
if not ablation_df.empty and {'answer_f1', 'parity_diff_geo'}.issubset(ablation_df.columns):
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=ablation_df, x='parity_diff_geo', y='answer_f1', hue='experiment', s=120)
    for _, row in ablation_df.iterrows():
        plt.text(row['parity_diff_geo'], row['answer_f1'], row['experiment'])
    plt.title('Fairness-Accuracy Tradeoff (Geo)')
    plt.xlabel('Demographic Parity Difference (lower is better)')
    plt.ylabel('Answer F1')
    plt.tight_layout()
    plt.show()